# 4 . Heikin/EMA Trend-Runner + Leverage

The specific idea to test: **wait until 1-2 candles form fully above the Nth EMA, go long, then let
it run** while momentum holds (bull Heikin-Ashi / price not fully below a shorter EMA); cut on a
fixed target if it goes choppy. **Stop = the previous swing low below the entry EMA, capped at x% of
equity.** Then scale it with leverage: $1000 account, up to 1:2000.

Expressed with `EmaCross`: `entry_mode='full_candle'`, `confirm_n=1..2`, `exit_mode='below_ema'`
(ride until a full candle forms below a shorter EMA) or `'ha_flip'`, structure stop via `ref_col`.


## Setup


In [1]:
import sys, os, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..')); import quant
import dataclasses, pandas as pd
from quant.data import get_ohlcv
from quant.engine import BacktestConfig
from quant.strategies import EmaCross
from quant import analytics as A, reporting as R
from quant.viz import ResearchChart, equity_chart
from experiments.base import Experiment

SYMBOL, SOURCE, MARKET = 'PAXGUSDT', 'binance', 'spot'
TF, TZ = '5m', 'UTC'
START, END = '2025-06-01', '2026-05-31'
df = get_ohlcv(SYMBOL, TF, start=START, end=END, source=SOURCE, market=MARKET, tz=TZ)
print(f'{len(df):,} {TF} bars')

[05:45:56] INFO | quant.data | cache HIT  binance PAXGUSDT 5m -> pushdown load


104,833 5m bars


## The strategy + a leveraged config
Long-only trend runner. Stop = previous swing low (structure), buffered, and capped so risk per
trade <= `sl_max_ref_risk_pct`. Leverage models real margin + stop-out (reason `margin_call`).


In [2]:
runner = EmaCross(ema_period=100, entry_mode='full_candle', confirm_n=2,
                  allow_long=True, allow_short=False, exit_mode='below_ema', exit_ema=20)

# $1000 account, 1:2000 leverage, gold 1 lot = 100 oz; risk-based sizing, structure stop.
lev_cfg = BacktestConfig(
    initial_cash=1000, spread=0.12, slippage_bps=1.0,
    margin_enabled=True, leverage=2000, contract_size=100, stop_out_level=50.0,
    sizing_mode='risk_pct_equity', sizing_value=1.0,          # risk 1% of equity per trade
    exit_enabled=True, sl_mode='ref_col', sl_ref_long_col='swing_last_low',
    sl_buffer_pct=0.05, sl_max_ref_risk_pct=1.5,              # stop no wider than 1.5%
    tp_mode='none',                                          # let the exit_mode ride it
    allow_rule_close=True,
)
res = runner.backtest(df, lev_cfg)
R.print_summary(res, df=df)
print('margin-call exits:', int((res.trades['close_reason']=='margin_call').sum()))

[05:45:57] INFO | quant.engine | backtest done | bars=104,833 trades=8,010 final_cash=0.01 return=-100.00% dd=100.00% | 510.6 ms


 PERFORMANCE SUMMARY
  Total return %                    -100.00
  Final cash                           0.01
  Trades                               8010
  Win rate %                          18.11
  Profit factor                        0.35
  Expectancy / trade                -0.1248
  Sharpe                              -6.17
  Sortino                             -5.49
  Calmar                             -10.02
  Max drawdown %                     100.00
  Recovery factor                     -1.00
  Avg bars held                           6
  Total fees                           0.00

  Best weekday: Mon (pnl=-60.43, win=22.2%)
  Best session: tokyo (pnl=-327.73, n=2917)
margin-call exits: 0


## Find the best settings
Sweep the entry confirmation, the ride-exit, the stop cap, and the risk %. Rank by Sharpe (risk-
adjusted, which matters most under leverage).


In [3]:
def study(name, desc, strat_space, cfg_space, cfg, metric='sharpe', min_trades=25):
    exp = Experiment(name=name, description=desc, strategy_cls=EmaCross, base_cfg=cfg,
                     symbol=SYMBOL, tf=TF, start=START, end=END,
                     strategy_space=strat_space, cfg_space=cfg_space,
                     metric=metric, direction='max', min_trades=min_trades)
    return exp.run(df=df)
SHOW=['num_trades','total_return_pct','win_rate_pct','profit_factor','sharpe','max_drawdown_pct']
r = study('runner_settings', 'trend-runner entry/exit/stop search',
          strat_space={'ema_period':[50,100,200], 'entry_mode':['full_candle'], 'confirm_n':[1,2],
                       'allow_short':[False], 'exit_mode':['below_ema','ha_flip'], 'exit_ema':[20,50]},
          cfg_space={'sl_max_ref_risk_pct':[1.0,1.5,2.5], 'sizing_value':[1.0]}, cfg=lev_cfg)
r[['ema_period','confirm_n','exit_mode','exit_ema','sl_max_ref_risk_pct']+SHOW].head(12)

[05:46:13] INFO | experiments | experiment 'runner_settings': 24 strategy x 3 cfg = 72 trials | bars=104833
[05:46:15] INFO | experiments | experiment 'runner_settings' -> experiments\results\runner_settings (best sharpe=-3.2550)


,ema_period,confirm_n,exit_mode,exit_ema,sl_max_ref_risk_pct,num_trades,total_return_pct,win_rate_pct,profit_factor,sharpe,max_drawdown_pct
0,100,2,below_ema,50,2.5,3616.0,-99.740014,16.758850,0.363480,-3.255035,99.769802
1,100,2,below_ema,50,1.5,3617.0,-99.745425,16.754216,0.363206,-3.266887,99.773266
2,100,2,below_ema,50,1.0,3617.0,-99.747039,16.754216,0.363046,-3.267523,99.774062
3,100,1,below_ema,50,2.5,4112.0,-99.924429,16.512646,0.371991,-3.326170,99.931510
4,100,1,below_ema,50,1.0,4114.0,-99.926702,16.504618,0.371564,-3.339146,99.933179
5,100,1,below_ema,50,1.5,4114.0,-99.927232,16.504618,0.371725,-3.346097,99.933663
6,200,2,below_ema,50,2.5,4463.0,-100.000021,16.132646,0.313875,-3.445080,100.000020
7,200,2,below_ema,50,1.5,4463.0,-100.000021,16.132646,0.313875,-3.449744,100.000019
8,200,2,below_ema,50,1.0,4464.0,-100.000021,16.129032,0.313875,-3.454225,100.000019
9,50,2,below_ema,50,1.0,1986.0,-98.784757,20.543807,0.327791,-3.519737,98.849642


## Scale with leverage
Leverage does NOT change the % return of a *risk-%-sized* strategy - it changes how much margin is
tied up and how close you run to a **stop-out**. What actually scales your P&L is **risk per trade**
(`sizing_value`). Higher risk% = bigger swings and deeper drawdowns. Sweep risk% at 1:2000 and watch
drawdown + margin-call count, not just return.


In [4]:
best = r.iloc[0]
chosen = EmaCross(ema_period=int(best['ema_period']), entry_mode='full_candle',
                  confirm_n=int(best['confirm_n']), allow_long=True, allow_short=False,
                  exit_mode=best['exit_mode'], exit_ema=int(best['exit_ema']))
rows=[]
for risk in [0.5,1.0,2.0,3.0,5.0]:
    c = dataclasses.replace(lev_cfg, sizing_value=risk, sl_max_ref_risk_pct=float(best['sl_max_ref_risk_pct']))
    st = chosen.backtest(df, c)
    stats = st.stats; mc = int((st.trades['close_reason']=='margin_call').sum())
    rows.append({'risk%/trade':risk,'return%':round(stats['total_return_pct'],1),
                 'maxDD%':round(stats['max_drawdown_pct'],1),'sharpe':round(stats['sharpe'],2),
                 'trades':int(stats['num_trades']),'margin_calls':mc,'final_$':round(stats['final_cash'],0)})
pd.DataFrame(rows)

[05:46:24] INFO | quant.engine | backtest done | bars=104,833 trades=3,616 final_cash=54.48 return=-94.55% dd=95.15% | 11.8 ms
[05:46:24] INFO | quant.engine | backtest done | bars=104,833 trades=3,616 final_cash=2.60 return=-99.74% dd=99.77% | 12.1 ms
[05:46:24] INFO | quant.engine | backtest done | bars=104,833 trades=3,615 final_cash=0.01 return=-100.00% dd=100.00% | 11.8 ms
[05:46:24] INFO | quant.engine | backtest done | bars=104,833 trades=3,154 final_cash=-0.00 return=-100.00% dd=100.00% | 10.9 ms
[05:46:25] INFO | quant.engine | backtest done | bars=104,833 trades=3,152 final_cash=-0.00 return=-100.00% dd=100.00% | 11.3 ms


,risk%/trade,return%,maxDD%,sharpe,trades,margin_calls,final_$
0,0.5,-94.6,95.1,-3.45,3616,0,54.0
1,1.0,-99.7,99.8,-3.26,3616,0,3.0
2,2.0,-100.0,100.0,-1.51,3615,0,0.0
3,3.0,-100.0,100.0,-3.68,3154,2,-0.0
4,5.0,-100.0,100.0,-6.65,3152,1,-0.0


## Inspect & validate the chosen setup
Chart the trades, then validate on an earlier out-of-sample window.


In [7]:
view = chosen.backtest(df, dataclasses.replace(lev_cfg, sizing_value=float(best.get('sizing_value',1.0))))
ch = ResearchChart(df, candles=True); ch.add_ema(int(best['ema_period'])); ch.add_ema(int(best['exit_ema']))
ch.add_trades(view.trades); ch.show()

[06:15:11] INFO | quant.engine | backtest done | bars=104,833 trades=3,617 final_cash=2.55 return=-99.75% dd=99.77% | 16.0 ms


FigureWidgetResampler({
    'data': [{'line': {'width': 1.1},
              'mode': 'lines',
              'name': '<b style="color:sandybrown">[R]</b> EMA100 <i style="color:#fc9944">~3h</i>',
              'type': 'scattergl',
              'uid': '1f4fc0ce-216d-435e-8fb2-a72aea5a5d3c',
              'x': array([Timestamp('2025-06-01 00:00:00+0000', tz='UTC'),
                          Timestamp('2025-06-01 00:05:00+0000', tz='UTC'),
                          Timestamp('2025-06-01 03:35:00+0000', tz='UTC'), ...,
                          Timestamp('2026-05-30 18:40:00+0000', tz='UTC'),
                          Timestamp('2026-05-30 20:30:00+0000', tz='UTC'),
                          Timestamp('2026-05-31 00:00:00+0000', tz='UTC')],
                         shape=(2500,), dtype=object),
              'y': {'bdata': ('AAAAAAAA+H8AAAAAAAD4fwAAAAAAAP' ... 'RRbbCxQP4X8KJ7r7FA0OcVKlitsUA='),
                    'dtype': 'f8'}},
             {'line': {'width': 1.1},
              'mode': 

In [6]:
oos = get_ohlcv(SYMBOL, TF, start='2025-01-01', end='2025-05-31', source=SOURCE, market=MARKET, tz=TZ)
ro = chosen.backtest(oos, lev_cfg)
print('OUT-OF-SAMPLE  sharpe=%.2f  return=%.1f%%  maxDD=%.1f%%  trades=%d  margin_calls=%d' % (
    ro.stats['sharpe'], ro.stats['total_return_pct'], ro.stats['max_drawdown_pct'],
    ro.stats['num_trades'], int((ro.trades['close_reason']=='margin_call').sum())))

[05:46:30] INFO | quant.data | cache HIT  binance PAXGUSDT 5m -> pushdown load
[05:46:30] INFO | quant.engine | backtest done | bars=43,201 trades=1,134 final_cash=132.02 return=-86.80% dd=86.80% | 6.2 ms


OUT-OF-SAMPLE  sharpe=-6.95  return=-86.8%  maxDD=86.8%  trades=1134  margin_calls=0


## Leverage & risk - what to focus on
- **Return % is set by risk-per-trade, not leverage.** With risk-%-of-equity sizing, 1:100 and
  1:2000 give the *same* trades until margin runs out - leverage only raises the ceiling on size.
- **Watch `max_drawdown_pct` and `margin_calls`.** High leverage + high risk% -> a losing streak can
  hit the **stop-out** (`stop_out_level`) and wipe the account. Aim for **0 margin calls** in-sample.
- **The structure stop caps risk per trade** (`sl_max_ref_risk_pct`); combined with `sizing_value`
  it bounds worst-case loss per trade. Keep risk small (0.5-2%) at high leverage.
- **Judge by Sharpe / Calmar and drawdown**, then out-of-sample. A setup that only survives at zero
  cost or blows up OOS is not real. 'Avoid chop' = the session/time filters from notebook 3 (Q4) and
  higher-timeframe bias (Q6) - add `session=`/`htf=` to `chosen` to trade only clean hours/trends.
